# Austria High-Growth Firms Analysis

This notebook analyzes the classified Austria firm data to understand high-growth firm patterns across industries, regions, and size bands.

In [1]:
import pandas as pd
import numpy as np

# Load and process data inline to avoid pickle compatibility issues
print("Loading raw data...")
df = pd.read_pickle('../data/raw/AT.pkl')

# Rename columns (from data cleaning notebook)
column_rename_map = {
    'Company name Latin alphabet': 'company_name',
    'Country ISO code': 'country_code', 
    'City\nLatin Alphabet': 'city',
    'NACE Rev. 2, core code (4 digits)': 'nace_code',
    'BvD ID number': 'bvd_id',
    'NACE Rev. 2 main section': 'nace_section',
    'Region in country': 'region_raw',
    'Status': 'status',
    'Date of incorporation': 'incorporation_date'
}

df = df.rename(columns=column_rename_map)

# Add employee columns
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
for year in years:
    raw_col = f'Number of employees\n{year}'
    num_col = f'emp_{year}_num'
    if raw_col in df.columns:
        df[num_col] = pd.to_numeric(df[raw_col], errors='coerce')

# Add founded_year
founded_col = 'Founded Year'
if founded_col in df.columns:
    df['founded_year'] = pd.to_datetime(df[founded_col], errors='coerce').dt.year

# Clean region
region_col = 'Region in country clean'
if region_col in df.columns:
    df['region'] = df[region_col].str.strip()

print(f"Data loaded and cleaned: {df.shape}")
print(f"Columns: {len(df.columns)}")
print("Available columns:", df.columns.tolist())

# Calculate growth variables
print("\nCalculating growth variables...")

# Growth calculations with "n.a." handling
def safe_growth(emp_current, emp_previous):
    """Calculate growth rate, returning 'n.a.' if either input is missing"""
    if pd.isna(emp_current) or pd.isna(emp_previous) or emp_previous == 0:
        return 'n.a.'
    return (emp_current - emp_previous) / emp_previous

# Calculate year-over-year growth
df['growth_2024'] = df.apply(lambda row: safe_growth(row['emp_2024_num'], row['emp_2023_num']), axis=1)
df['growth_2023'] = df.apply(lambda row: safe_growth(row['emp_2023_num'], row['emp_2022_num']), axis=1)  
df['growth_2022'] = df.apply(lambda row: safe_growth(row['emp_2022_num'], row['emp_2021_num']), axis=1)

# Calculate AAGR (average annual growth rate)
def calculate_aagr(growth_2024, growth_2023, growth_2022):
    """Calculate AAGR from three growth rates, returning 'n.a.' if any input is missing"""
    growths = [growth_2024, growth_2023, growth_2022]
    if any(g == 'n.a.' for g in growths):
        return 'n.a.'
    # AAGR formula: ((1 + g1) * (1 + g2) * (1 + g3))^(1/3) - 1
    product = 1
    for g in growths:
        product *= (1 + g)
    return product**(1/3) - 1

df['aagr_2024'] = df.apply(lambda row: calculate_aagr(row['growth_2024'], row['growth_2023'], row['growth_2022']), axis=1)

print("Growth variables calculated")

# Apply Consistent High Growth Firm classification
print("\nApplying classification...")

# Convert growth columns to numeric for classification (keeping 'n.a.' as NaN)
growth_cols = ['growth_2024', 'growth_2023', 'growth_2022', 'aagr_2024']
for col in growth_cols:
    df[f'{col}_num'] = pd.to_numeric(df[col], errors='coerce')

# Classification logic
def classify_high_growth_firm(row):
    """
    Classify as Consistent High Growth Firm following Belgium methodology
    Returns: 1 (high growth), 0 (not high growth), 'n.a.' (insufficient data)
    """
    # Check if firm has minimum size (10+ employees in 2021)
    if pd.isna(row['emp_2021_num']) or row['emp_2021_num'] < 10:
        return 'n.a.'
    
    # Check if all required growth data is available
    required_growth = [row['growth_2024_num'], row['growth_2023_num'], row['growth_2022_num']]
    if any(pd.isna(g) for g in required_growth):
        return 'n.a.'
    
    # High growth criteria: average growth > 10% over 3 years
    avg_growth = sum(required_growth) / len(required_growth)
    if avg_growth > 0.10:
        return 1
    else:
        return 0

df['ConsistentHighGrowthFirm_2024'] = df.apply(classify_high_growth_firm, axis=1)

print("Classification applied")
print(f"Final dataset: {df.shape}")
print(f"Columns: {len(df.columns)}")

Loading raw data...
Data loaded and cleaned: (46085, 30)
Columns: 30
Available columns: ['Unnamed: 0', 'company_name', 'country_code', 'city', 'nace_code', 'bvd_id', 'nace_section', 'region_raw', 'status', 'incorporation_date', 'Number of employees\n2024', 'Number of employees\n2023', 'Number of employees\n2022', 'Number of employees\n2021', 'Number of employees\n2020', 'Number of employees\n2019', 'Number of employees\n2018', 'Number of employees\n2017', 'Founded Year', 'Region in country clean', 'emp_2017_num', 'emp_2018_num', 'emp_2019_num', 'emp_2020_num', 'emp_2021_num', 'emp_2022_num', 'emp_2023_num', 'emp_2024_num', 'founded_year', 'region']

Calculating growth variables...
Growth variables calculated

Applying classification...
Classification applied
Final dataset: (46085, 39)
Columns: 39


## Table 1: Overall Growth-Class Distribution

Distribution of firms by their high-growth classification status.

In [2]:
# Table 1: Overall growth-class distribution
total_firms = len(df)
classified_firms = (df['ConsistentHighGrowthFirm_2024'] != 'n.a.').sum()
unclassified_firms = (df['ConsistentHighGrowthFirm_2024'] == 'n.a.').sum()

# Among classified firms
classified_df = df[df['ConsistentHighGrowthFirm_2024'] != 'n.a.']
high_growth_firms = (classified_df['ConsistentHighGrowthFirm_2024'] == 1).sum()
not_high_growth_firms = (classified_df['ConsistentHighGrowthFirm_2024'] == 0).sum()

print("Table 1: Overall Growth-Class Distribution")
print("=" * 50)
print(f"Total firms: {total_firms:,}")
print(f"Classified firms: {classified_firms:,} ({classified_firms/total_firms:.1%})")
print(f"Unclassified firms (n.a.): {unclassified_firms:,} ({unclassified_firms/total_firms:.1%})")
print()
print("Among classified firms:")
print(f"High-growth firms (1): {high_growth_firms:,} ({high_growth_firms/classified_firms:.1%})")
print(f"Not high-growth firms (0): {not_high_growth_firms:,} ({not_high_growth_firms/classified_firms:.1%})")

Table 1: Overall Growth-Class Distribution
Total firms: 46,085
Classified firms: 18,629 (40.4%)
Unclassified firms (n.a.): 27,456 (59.6%)

Among classified firms:
High-growth firms (1): 1,883 (10.1%)
Not high-growth firms (0): 16,746 (89.9%)


## Table 2: Industry Summary

Summary statistics by industry: number of firms, mean growth (AAGR), and share of high-growth firms.

In [3]:
# Table 2: Industry summary
if 'nace_section' in df.columns:
    # Group by industry (NACE section)
    industry_summary = df.groupby('nace_section').agg(
        total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
        classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
        high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum()),
        mean_aagr=('aagr_2024', lambda x: x[x != 'n.a.'].astype(float).mean())
    )
    
    # Calculate share of high-growth firms (among classified)
    industry_summary['high_growth_share'] = industry_summary['high_growth_firms'] / industry_summary['classified_firms']
    
    # Sort by number of firms descending and show top 10
    industry_summary = industry_summary.sort_values('total_firms', ascending=False).head(10)
    
    print("Table 2: Industry Summary (by NACE Section) - Top 10")
    print("=" * 80)
    print(industry_summary.to_string(float_format='%.3f'))
else:
    print("Industry column not found in dataset")

Table 2: Industry Summary (by NACE Section) - Top 10
                                                                          total_firms  classified_firms  high_growth_firms  mean_aagr  high_growth_share
nace_section                                                                                                                                            
G - Wholesale and retail trade; repair of motor vehicles and motorcycles         9700              4510                366      0.032              0.081
F - Construction                                                                 7564              3662                340      0.041              0.093
C - Manufacturing                                                                6862              3768                330      0.036              0.088
I - Accommodation and food service activities                                    4989              1171                142      0.070              0.121
M - Professional, scientific 

## Table 3: Region Summary

Summary statistics by region: number of firms, mean growth (AAGR), and share of high-growth firms.

In [4]:
# Table 3: Region summary
if 'region' in df.columns:
    # Group by region
    region_summary = df.groupby('region').agg(
        total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
        classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
        high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum()),
        mean_aagr=('aagr_2024', lambda x: x[x != 'n.a.'].astype(float).mean())
    )
    
    # Calculate share of high-growth firms (among classified)
    region_summary['high_growth_share'] = region_summary['high_growth_firms'] / region_summary['classified_firms']
    
    # Sort by number of firms descending
    region_summary = region_summary.sort_values('total_firms', ascending=False)
    
    print("Table 3: Region Summary")
    print("=" * 80)
    print(region_summary.to_string(float_format='%.3f'))
else:
    print("Region column not found in dataset")

Table 3: Region Summary
                  total_firms  classified_firms  high_growth_firms  mean_aagr  high_growth_share
region                                                                                          
Wien                    10066              3108                371      0.067              0.119
Oberosterreich           7884              3896                404      0.047              0.104
Niederosterreich         7118              3047                296      0.035              0.097
Steiermark               5966              2455                255      0.038              0.104
Tirol                    4630              1794                153      0.029              0.085
Salzburg                 3935              1711                178      0.026              0.104
Karnten                  2691              1108                100      0.052              0.090
Vorarlberg               2290               916                 68      0.033              0.074
Burgen

## Table 4: Size-Band Summary

Summary statistics by size band: number of firms, mean growth (AAGR), and share of high-growth firms.

In [5]:
# Table 4: Size-band summary
# Create size bands from emp_2021_num
df_temp = df.copy()
df_temp['size_band'] = pd.cut(df_temp['emp_2021_num'], 
                              bins=[10, 19, 49, 249, float('inf')], 
                              labels=['11–19', '20–49', '50–249', '250+'],
                              right=True)

size_summary = df_temp.groupby('size_band').agg(
    total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
    classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
    high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum()),
    mean_aagr=('aagr_2024', lambda x: x[x != 'n.a.'].astype(float).mean())
)

# Calculate share of high-growth firms (among classified)
size_summary['high_growth_share'] = size_summary['high_growth_firms'] / size_summary['classified_firms']

# Sort by size band order
size_order = ['11–19', '20–49', '50–249', '250+']
size_summary = size_summary.reindex(size_order)

print("Table 4: Size-Band Summary")
print("=" * 80)
print(size_summary.to_string(float_format='%.3f'))

Table 4: Size-Band Summary
           total_firms  classified_firms  high_growth_firms  mean_aagr  high_growth_share
size_band                                                                                
11–19            10890              5876                671      0.003              0.114
20–49            10781              6389                643     -0.002              0.101
50–249            5532              3609                333     -0.004              0.092
250+              1426              1045                 65      0.002              0.062


/var/folders/4_/pht_r21d0yg70lpf8ry01d300000gn/T/ipykernel_38567/615017801.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  size_summary = df_temp.groupby('size_band').agg(


## Key Observations

Based on the analysis of Austria's high-growth firms, here are the 3 core observations:

### 1. **Vienna Dominates High-Growth Entrepreneurship**
Vienna stands out with the highest share of high-growth firms (11.9%) among all Austrian regions, significantly outperforming other regions like Burgenland (7.4%) and Tirol (8.5%). This suggests the capital region provides unique advantages for entrepreneurial growth, potentially through better access to finance, talent, and markets.

### 2. **Smaller Firms Drive Growth Potential**  
High-growth potential is concentrated in smaller firms, with companies employing 11-19 people showing an 11.4% high-growth rate - nearly double that of large firms (250+ employees) at just 6.2%. This pattern indicates that Austria's entrepreneurial dynamism comes from agile, smaller businesses rather than established corporations.

### 3. **Growth is Highly Concentrated**
Only 10.1% of classifiable Austrian firms achieve high-growth status, with 1,883 out of 18,629 eligible firms meeting the >10% average annual growth threshold. This concentration highlights that exceptional growth remains rare, even in a developed economy like Austria's.

## Final Visualizations for Press Release

Create the 3 key visualizations that will be included in the press release.

### Visual 1: Overall Growth-Class Distribution

import matplotlib.pyplot as plt
import os

# Create outputs/figures directory if it doesn't exist
os.makedirs('../outputs/figures', exist_ok=True)

# Data for the chart
total_firms = len(df)
classified_firms = (df['ConsistentHighGrowthFirm_2024'] != 'n.a.').sum()
unclassified_firms = total_firms - classified_firms
high_growth_firms = (df['ConsistentHighGrowthFirm_2024'] == 1).sum()
not_high_growth_firms = (df['ConsistentHighGrowthFirm_2024'] == 0).sum()

# Create the visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left chart: Overall classification status
ax1.bar(['Classified', 'Unclassified'], [classified_firms, unclassified_firms], 
        color=['#1f77b4', '#ff7f0e'], alpha=0.7)
ax1.set_title('Firm Classification Status', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Firms', fontsize=12)
ax1.grid(axis='y', alpha=0.3)

# Add percentage labels
for i, v in enumerate([classified_firms, unclassified_firms]):
    ax1.text(i, v + 500, f'{v:,}\n({v/total_firms:.1%})', ha='center', va='bottom', fontsize=10)

# Right chart: Distribution among classified firms
ax2.bar(['High-Growth', 'Not High-Growth'], [high_growth_firms, not_high_growth_firms], 
        color=['#2ca02c', '#d62728'], alpha=0.7)
ax2.set_title('Distribution Among Classified Firms', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Firms', fontsize=12)
ax2.grid(axis='y', alpha=0.3)

# Add percentage labels
for i, v in enumerate([high_growth_firms, not_high_growth_firms]):
    ax2.text(i, v + 50, f'{v:,}\n({v/classified_firms:.1%})', ha='center', va='bottom', fontsize=10)

plt.suptitle('Austria High-Growth Firms: Overall Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
print("Saved: ../outputs/figures/growth_class_distribution.png")

### Visual 2: Industry Growth Comparison

# Get top 10 industries by total firms
if 'nace_section' in df.columns:
    industry_stats = df.groupby('nace_section').agg(
        total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
        classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
        high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum())
    ).sort_values('total_firms', ascending=False).head(10)
    
    industry_stats['high_growth_share'] = industry_stats['high_growth_firms'] / industry_stats['classified_firms']
    
    # Create horizontal bar chart
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Clean up industry names (take first letter and main description)
    industry_names = [name.split(' - ')[1][:30] + '...' if len(name.split(' - ')) > 1 and len(name.split(' - ')[1]) > 30 
                     else name.split(' - ')[1] if len(name.split(' - ')) > 1 
                     else name[:30] + '...' if len(name) > 30 else name 
                     for name in industry_stats.index]
    
    bars = ax.barh(industry_names, industry_stats['high_growth_share'] * 100, 
                   color='#1f77b4', alpha=0.7)
    
    ax.set_title('High-Growth Firm Share by Industry (Top 10 by Firm Count)', 
                fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('High-Growth Firm Share (%)', fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels on bars
    for bar, share in zip(bars, industry_stats['high_growth_share']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
               f'{share:.1%}', ha='left', va='center', fontsize=10, fontweight='bold')
    
    # Add firm count labels
    for i, (bar, total) in enumerate(zip(bars, industry_stats['total_firms'])):
        ax.text(0.5, bar.get_y() + bar.get_height()/2, 
               f'{total:,} firms', ha='left', va='center', fontsize=9, alpha=0.8)
    
    plt.tight_layout()
    print("Saved: ../outputs/figures/industry_growth_comparison.png")
else:
    print("Industry column not found")

### Visual 3: Regional Growth Comparison

# Get region statistics
if 'region' in df.columns:
    region_stats = df.groupby('region').agg(
        total_firms=('ConsistentHighGrowthFirm_2024', 'size'),
        classified_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x != 'n.a.').sum()),
        high_growth_firms=('ConsistentHighGrowthFirm_2024', lambda x: (x == 1).sum())
    ).sort_values('total_firms', ascending=False)
    
    region_stats['high_growth_share'] = region_stats['high_growth_firms'] / region_stats['classified_firms']
    
    # Create horizontal bar chart
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Sort by high-growth share for better visualization
    region_stats_sorted = region_stats.sort_values('high_growth_share', ascending=True)
    
    bars = ax.barh(range(len(region_stats_sorted)), region_stats_sorted['high_growth_share'] * 100, 
                   color='#2ca02c', alpha=0.7)
    
    ax.set_title('High-Growth Firm Share by Region', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('High-Growth Firm Share (%)', fontsize=12)
    ax.set_yticks(range(len(region_stats_sorted)))
    ax.set_yticklabels(region_stats_sorted.index)
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels on bars
    for bar, share in zip(bars, region_stats_sorted['high_growth_share']):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, 
               f'{share:.1%}', ha='left', va='center', fontsize=10, fontweight='bold')
    
    # Add firm count labels
    for i, (bar, total) in enumerate(zip(bars, region_stats_sorted['total_firms'])):
        ax.text(0.2, bar.get_y() + bar.get_height()/2, 
               f'{total:,} firms', ha='left', va='center', fontsize=9, alpha=0.8)
    
    # Highlight Vienna
    vienna_idx = list(region_stats_sorted.index).index('Wien')
    bars[vienna_idx].set_color('#ff7f0e')
    ax.text(bars[vienna_idx].get_width() + 0.05, bars[vienna_idx].get_y() + bars[vienna_idx].get_height()/2,
           f'{region_stats_sorted.loc["Wien", "high_growth_share"]:.1%} ← Vienna', 
           ha='left', va='center', fontsize=11, fontweight='bold', color='#ff7f0e')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/region_growth_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Saved: ../outputs/figures/region_growth_comparison.png")
else:
    print("Region column not found")